# Notebook 08 — Time Regime Analysis

**Assignment**: Regime family — day-over-day stability, intraday seasonality, volatility clustering.

**Lenses**:
1. Day-over-day stability: per-product mean / std / AR(1) for days 2, 3, 4 — flag stable vs drifting
2. Intraday seasonality: time-of-day (tick-bucket) mean patterns, FFT periodicity
3. Volatility clustering: squared-returns autocorrelation (ARCH-like evidence)

**Data**: `data/round_5/prices/prices_round_5_day_{2,3,4}.csv` (semicolon-separated, READ ONLY)

**Output criteria (triage rubric from AGENT_BRIEF)**:
- `likely exploitable`: stable stats across all 3 days AND (|AR(1)| > 0.3 OR detectable FFT peak OR hardcoded FV per other notebooks)
- `probably tradable`: mostly stable but some drift OR moderate AR(1) 0.1–0.3
- `probably noise`: drifting stats, AR(1) near 0, no intraday structure

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

BASE = '/Users/bensinek/Documents/Coding/Prosperity4/data/round_5/prices'

days = [2, 3, 4]
dfs = {}
for d in days:
    dfs[d] = pd.read_csv(f'{BASE}/prices_round_5_day_{d}.csv', sep=';')

# Stack all days
all_df = pd.concat(dfs.values(), ignore_index=True)
PRODUCTS = sorted(all_df['product'].unique())
CATEGORIES = {
    'GALAXY_SOUNDS': [p for p in PRODUCTS if p.startswith('GALAXY_SOUNDS')],
    'SLEEP_POD':     [p for p in PRODUCTS if p.startswith('SLEEP_POD')],
    'MICROCHIP':     [p for p in PRODUCTS if p.startswith('MICROCHIP')],
    'PEBBLES':       [p for p in PRODUCTS if p.startswith('PEBBLES')],
    'ROBOT':         [p for p in PRODUCTS if p.startswith('ROBOT')],
    'UV_VISOR':      [p for p in PRODUCTS if p.startswith('UV_VISOR')],
    'TRANSLATOR':    [p for p in PRODUCTS if p.startswith('TRANSLATOR')],
    'PANEL':         [p for p in PRODUCTS if p.startswith('PANEL')],
    'OXYGEN_SHAKE':  [p for p in PRODUCTS if p.startswith('OXYGEN_SHAKE')],
    'SNACKPACK':     [p for p in PRODUCTS if p.startswith('SNACKPACK')],
}

print(f'Products: {len(PRODUCTS)}, Days: {days}')
print(f'Ticks per day: {len(dfs[2]["timestamp"].unique())}')
for cat, prods in CATEGORIES.items():
    print(f'  {cat}: {len(prods)} products')

Products: 50, Days: [2, 3, 4]
Ticks per day: 10000
  GALAXY_SOUNDS: 5 products
  SLEEP_POD: 5 products
  MICROCHIP: 5 products
  PEBBLES: 5 products
  ROBOT: 5 products
  UV_VISOR: 5 products
  TRANSLATOR: 5 products
  PANEL: 5 products
  OXYGEN_SHAKE: 5 products
  SNACKPACK: 5 products


## 1. Day-over-day stability: mean, std, AR(1) per product per day

For each product and each of days 2/3/4, compute:
- `mean`: mean of mid_price
- `std`: std of mid_price
- `ar1`: AR(1) of log-returns (regress r_t on r_{t-1})

Then measure cross-day stability via coefficient of variation of the mean, and range of std and ar1.

In [2]:
def compute_ar1(series):
    """AR(1) of log-returns: regress r_t on r_{t-1}. Returns rho."""
    prices = series.dropna().values
    if len(prices) < 10:
        return np.nan
    # Use price returns (first differences of log price)
    rets = np.diff(np.log(prices + 1e-9))
    if len(rets) < 4:
        return np.nan
    rho, _, _, _, _ = np.polyfit(rets[:-1], rets[1:], 1, full=False), None, None, None, None
    # simple OLS
    x = rets[:-1]
    y = rets[1:]
    rho = np.corrcoef(x, y)[0, 1]
    return rho

records = []
for prod in PRODUCTS:
    for d in days:
        sub = dfs[d][dfs[d]['product'] == prod].sort_values('timestamp')
        mids = sub['mid_price']
        records.append({
            'product': prod,
            'day': d,
            'mean': mids.mean(),
            'std': mids.std(),
            'ar1': compute_ar1(mids),
            'n_ticks': len(mids),
        })

stats_df = pd.DataFrame(records)
print(stats_df.head(15))

                          product  day         mean         std       ar1  \
0       GALAXY_SOUNDS_BLACK_HOLES    2  10680.49205  558.210674 -0.017677   
1       GALAXY_SOUNDS_BLACK_HOLES    3  11107.87780  323.321687 -0.005689   
2       GALAXY_SOUNDS_BLACK_HOLES    4  12612.24640  529.657953 -0.026802   
3       GALAXY_SOUNDS_DARK_MATTER    2  10111.91380  250.268983  0.002212   
4       GALAXY_SOUNDS_DARK_MATTER    3  10421.95090  320.140931 -0.023754   
5       GALAXY_SOUNDS_DARK_MATTER    4  10146.12075  324.327016 -0.012872   
6   GALAXY_SOUNDS_PLANETARY_RINGS    2  10013.25240  480.558543 -0.004414   
7   GALAXY_SOUNDS_PLANETARY_RINGS    3  11615.27320  270.848939 -0.003136   
8   GALAXY_SOUNDS_PLANETARY_RINGS    4  10671.49395  397.976794 -0.002086   
9      GALAXY_SOUNDS_SOLAR_FLAMES    2  11095.64355  496.853297 -0.024048   
10     GALAXY_SOUNDS_SOLAR_FLAMES    3  11259.99860  443.693310 -0.002949   
11     GALAXY_SOUNDS_SOLAR_FLAMES    4  10922.07295  327.266056 -0.009023   

In [3]:
# Pivot to wide and compute stability metrics
pivot_mean = stats_df.pivot(index='product', columns='day', values='mean')
pivot_std  = stats_df.pivot(index='product', columns='day', values='std')
pivot_ar1  = stats_df.pivot(index='product', columns='day', values='ar1')

stability = pd.DataFrame(index=PRODUCTS)
# CV of mean (coefficient of variation across days — measures relative drift)
stability['mean_cv'] = pivot_mean.std(axis=1) / pivot_mean.mean(axis=1).abs()
# Range of std across days
stability['std_range'] = pivot_std.max(axis=1) - pivot_std.min(axis=1)
# Range of ar1 across days
stability['ar1_range'] = pivot_ar1.max(axis=1) - pivot_ar1.min(axis=1)
# Mean ar1 across days
stability['ar1_mean'] = pivot_ar1.mean(axis=1)
# Mean std across days
stability['std_mean'] = pivot_std.mean(axis=1)
# Mean of mean across days
stability['price_mean'] = pivot_mean.mean(axis=1)

# Add individual day columns for inspection
for d in days:
    stability[f'mean_d{d}'] = pivot_mean[d]
    stability[f'std_d{d}']  = pivot_std[d]
    stability[f'ar1_d{d}']  = pivot_ar1[d]

# Classify stability
def classify_stability(row):
    # Stable: low mean drift (CV < 0.005) and ar1 consistent (range < 0.1)
    if row['mean_cv'] < 0.005 and row['ar1_range'] < 0.1:
        return 'STABLE'
    elif row['mean_cv'] < 0.02 and row['ar1_range'] < 0.2:
        return 'MOSTLY_STABLE'
    else:
        return 'DRIFTING'

stability['stability'] = stability.apply(classify_stability, axis=1)

print('\nStability classification counts:')
print(stability['stability'].value_counts())
print('\nStable products:')
print(stability[stability['stability']=='STABLE'][['price_mean','std_mean','ar1_mean','mean_cv','ar1_range']].to_string())


Stability classification counts:
stability
DRIFTING         41
MOSTLY_STABLE     8
STABLE            1
Name: count, dtype: int64

Stable products:
                       price_mean    std_mean  ar1_mean   mean_cv  ar1_range
SNACKPACK_RASPBERRY  10077.812067  166.987958 -0.017045  0.003474   0.017325


In [4]:
print('\nDrifting products:')
print(stability[stability['stability']=='DRIFTING'][['price_mean','std_mean','ar1_mean','mean_cv','ar1_range']].sort_values('mean_cv', ascending=False).to_string())

print('\nAll products sorted by mean_cv:')
print(stability[['price_mean','std_mean','ar1_mean','mean_cv','ar1_range','stability']].sort_values('mean_cv').to_string())


Drifting products:
                                 price_mean     std_mean  ar1_mean   mean_cv  ar1_range
MICROCHIP_OVAL                  8179.598717   488.599027 -0.007261  0.219664   0.014291
PEBBLES_XS                      7404.639767   602.058657 -0.017158  0.217642   0.015895
MICROCHIP_SQUARE               13594.748300   772.154518 -0.022137  0.148741   0.022551
UV_VISOR_AMBER                  7911.696250   282.665422 -0.003124  0.147202   0.004758
PEBBLES_XL                     13225.589317  1167.838035  0.009087  0.118783   0.016679
PEBBLES_S                       8932.356733   408.646334  0.008308  0.098647   0.017594
MICROCHIP_TRIANGLE              9686.391083   339.243851 -0.007766  0.095917   0.015647
SLEEP_POD_POLYESTER            11840.560950   431.653726 -0.001387  0.090128   0.013902
GALAXY_SOUNDS_BLACK_HOLES      11466.872083   470.396771 -0.016723  0.088488   0.021112
ROBOT_IRONING                   8701.570267   427.667547 -0.116738  0.088063   0.075410
SLEEP_POD_SU

In [5]:
# Plot: mean_cv vs ar1_mean scatter, colored by category
cat_colors = {
    'GALAXY_SOUNDS': '#e41a1c',
    'SLEEP_POD':     '#377eb8',
    'MICROCHIP':     '#4daf4a',
    'PEBBLES':       '#984ea3',
    'ROBOT':         '#ff7f00',
    'UV_VISOR':      '#a65628',
    'TRANSLATOR':    '#f781bf',
    'PANEL':         '#999999',
    'OXYGEN_SHAKE':  '#17becf',
    'SNACKPACK':     '#bcbd22',
}

def get_cat(prod):
    for cat in CATEGORIES:
        if prod in CATEGORIES[cat]:
            return cat
    return 'UNKNOWN'

stability['category'] = [get_cat(p) for p in stability.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for cat, color in cat_colors.items():
    mask = stability['category'] == cat
    sub = stability[mask]
    ax.scatter(sub['mean_cv'], sub['ar1_mean'], c=color, label=cat, alpha=0.8, s=60)
ax.axhline(0.3, color='red', linestyle='--', alpha=0.5, label='|AR1|=0.3 threshold')
ax.axhline(-0.3, color='red', linestyle='--', alpha=0.5)
ax.axvline(0.005, color='green', linestyle='--', alpha=0.5, label='CV=0.005 stability')
ax.set_xlabel('Mean CV across days (drift measure)')
ax.set_ylabel('Mean AR(1) of returns')
ax.set_title('Day-over-day stability vs AR(1)')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
for cat, color in cat_colors.items():
    mask = stability['category'] == cat
    sub = stability[mask]
    ax2.scatter(sub['std_mean'], sub['ar1_range'], c=color, label=cat, alpha=0.8, s=60)
ax2.axhline(0.1, color='red', linestyle='--', alpha=0.5, label='AR1 range=0.1')
ax2.set_xlabel('Mean std of mid_price')
ax2.set_ylabel('AR(1) range across days')
ax2.set_title('Volatility vs AR(1) consistency')
ax2.legend(fontsize=7, ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/08_stability_scatter.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved stability scatter')

Saved stability scatter


In [6]:
# Per-day AR(1) heatmap across all products
ar1_matrix = pivot_ar1.copy()
ar1_matrix.index = [p.replace('GALAXY_SOUNDS_', 'GS_').replace('SLEEP_POD_', 'SP_')
                    .replace('MICROCHIP_', 'MC_').replace('PEBBLES_', 'PB_')
                    .replace('ROBOT_', 'RB_').replace('UV_VISOR_', 'UV_')
                    .replace('TRANSLATOR_', 'TR_').replace('PANEL_', 'PN_')
                    .replace('OXYGEN_SHAKE_', 'OX_').replace('SNACKPACK_', 'SN_')
                    for p in ar1_matrix.index]

fig, ax = plt.subplots(figsize=(8, 14))
im = ax.imshow(ar1_matrix.values, cmap='RdBu', vmin=-0.5, vmax=0.5, aspect='auto')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels([f'Day {d}' for d in days])
ax.set_yticks(range(len(ar1_matrix)))
ax.set_yticklabels(ar1_matrix.index, fontsize=7)
ax.set_title('AR(1) of returns — per product per day\n(blue=positive, red=negative)')
plt.colorbar(im, ax=ax, label='AR(1) rho')
plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/08_ar1_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved AR(1) heatmap')

Saved AR(1) heatmap


## 2. Intraday seasonality: tick-bucket mean patterns

The trading day runs 10,000 ticks (timestamps 0–999900, step 100). Divide into 20 buckets of 500 ticks each. For each product, compute mean mid_price per bucket averaged across all 3 days. Look for systematic intraday curves.

In [7]:
# Build intraday bucket means
N_BUCKETS = 20
TICK_MAX = 999900

all_df2 = all_df.copy()
all_df2['bucket'] = pd.cut(all_df2['timestamp'], bins=N_BUCKETS, labels=False)

# Mean mid_price per product per bucket (averaged across days)
intraday_mean = all_df2.groupby(['product', 'bucket'])['mid_price'].mean().unstack('bucket')

# Normalize: subtract each product's own mean so we see the SHAPE, not the level
intraday_norm = intraday_mean.subtract(intraday_mean.mean(axis=1), axis=0)

# Measure intraday amplitude: max - min of normalized curve
intraday_amplitude = intraday_norm.max(axis=1) - intraday_norm.min(axis=1)
stability['intraday_amplitude'] = intraday_amplitude

print('Intraday amplitude (max-min of normalized bucket mean) — top 20:')
print(intraday_amplitude.sort_values(ascending=False).head(20))

Intraday amplitude (max-min of normalized bucket mean) — top 20:
product
PEBBLES_XL                     2667.370667
PEBBLES_XS                     1654.777000
MICROCHIP_SQUARE               1567.891000
OXYGEN_SHAKE_GARLIC            1498.688000
GALAXY_SOUNDS_BLACK_HOLES      1456.273333
MICROCHIP_OVAL                 1396.914333
PANEL_2X4                      1067.006667
PEBBLES_S                       998.680667
ROBOT_MOPPING                   960.477000
PEBBLES_M                       923.028000
ROBOT_IRONING                   917.280000
PANEL_1X4                       911.536333
PEBBLES_L                       891.882667
MICROCHIP_RECTANGLE             845.642333
UV_VISOR_AMBER                  844.207000
SLEEP_POD_SUEDE                 839.722000
OXYGEN_SHAKE_MORNING_BREATH     808.140333
OXYGEN_SHAKE_EVENING_BREATH     800.648667
SLEEP_POD_POLYESTER             791.236333
OXYGEN_SHAKE_CHOCOLATE          752.929000
dtype: float64


In [8]:
# Plot intraday profiles for top 10 highest-amplitude products
top_intraday = intraday_amplitude.sort_values(ascending=False).head(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, prod in enumerate(top_intraday):
    ax = axes[i]
    # Per-day intraday curves
    for d, col in zip(days, ['blue', 'orange', 'green']):
        sub = dfs[d][dfs[d]['product'] == prod].sort_values('timestamp').copy()
        sub['bucket'] = pd.cut(sub['timestamp'], bins=N_BUCKETS, labels=False)
        bucket_mean = sub.groupby('bucket')['mid_price'].mean()
        # Normalize
        bucket_norm = bucket_mean - bucket_mean.mean()
        ax.plot(bucket_norm.index, bucket_norm.values, color=col, alpha=0.7, label=f'Day {d}')
    ax.set_title(prod.replace('GALAXY_SOUNDS_', 'GS_').replace('SLEEP_POD_', 'SP_')
                 .replace('OXYGEN_SHAKE_', 'OX_').replace('TRANSLATOR_', 'TR_'), fontsize=8)
    ax.set_xlabel('Bucket')
    ax.set_ylabel('Δ mid')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

plt.suptitle('Intraday profiles — top 10 highest amplitude products\n(normalized, 3 days overlaid)', fontsize=12)
plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/08_intraday_top10.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved intraday top-10')

Saved intraday top-10


In [9]:
# FFT on each product's intraday mean curve (20 buckets) — look for dominant periodicities
from numpy.fft import rfft, rfftfreq

fft_records = []
for prod in PRODUCTS:
    curve = intraday_norm.loc[prod].dropna().values
    if len(curve) < 4:
        continue
    freqs = rfftfreq(len(curve))
    power = np.abs(rfft(curve))**2
    # Ignore DC (freq=0)
    if len(power) > 1:
        dominant_idx = np.argmax(power[1:]) + 1
        dominant_freq = freqs[dominant_idx]
        dominant_power = power[dominant_idx]
        total_power = power[1:].sum()
        frac_power = dominant_power / total_power if total_power > 0 else 0
        fft_records.append({
            'product': prod,
            'dominant_freq': dominant_freq,
            'dominant_period_buckets': 1.0 / dominant_freq if dominant_freq > 0 else np.nan,
            'dominant_power_frac': frac_power,
        })

fft_df = pd.DataFrame(fft_records).set_index('product')
print('\nFFT: products with dominant frequency capturing >50% of intraday power:')
print(fft_df[fft_df['dominant_power_frac'] > 0.5].sort_values('dominant_power_frac', ascending=False).to_string())
print('\nAll FFT results sorted by dominant_power_frac:')
print(fft_df.sort_values('dominant_power_frac', ascending=False).to_string())


FFT: products with dominant frequency capturing >50% of intraday power:
                               dominant_freq  dominant_period_buckets  dominant_power_frac
product                                                                                   
OXYGEN_SHAKE_CHOCOLATE                  0.05                     20.0             0.929735
PEBBLES_L                               0.05                     20.0             0.915076
UV_VISOR_ORANGE                         0.05                     20.0             0.912516
MICROCHIP_CIRCLE                        0.05                     20.0             0.895879
OXYGEN_SHAKE_EVENING_BREATH             0.05                     20.0             0.864031
ROBOT_MOPPING                           0.05                     20.0             0.857355
MICROCHIP_RECTANGLE                     0.05                     20.0             0.844761
PANEL_4X4                               0.05                     20.0             0.820736
PEBBLES_XL       

In [10]:
# Do the same FFT but on the RAW mid-price time-series (not bucketized) using day-level data
# Use only Day 2 for illustration (10,000 ticks = good resolution)

fft_raw_records = []
for prod in PRODUCTS:
    sub = dfs[2][dfs[2]['product'] == prod].sort_values('timestamp')['mid_price'].values
    if len(sub) < 100:
        continue
    # Detrend (subtract linear trend)
    detrended = sub - np.polyval(np.polyfit(np.arange(len(sub)), sub, 1), np.arange(len(sub)))
    freqs = rfftfreq(len(detrended))
    power = np.abs(rfft(detrended))**2
    # Find top 3 peaks (exclude DC)
    if len(power) > 3:
        top3_idx = np.argsort(power[1:])[-3:][::-1] + 1
        total_power = power[1:].sum()
        top1_frac = power[top3_idx[0]] / total_power if total_power > 0 else 0
        fft_raw_records.append({
            'product': prod,
            'top1_freq': freqs[top3_idx[0]],
            'top1_period_ticks': 1.0 / freqs[top3_idx[0]] if freqs[top3_idx[0]] > 0 else np.nan,
            'top1_power_frac': top1_frac,
        })

fft_raw_df = pd.DataFrame(fft_raw_records).set_index('product')
print('\nRaw FFT (day 2, detrended) — products with top-1 frequency capturing >30% of power:')
print(fft_raw_df[fft_raw_df['top1_power_frac'] > 0.3].sort_values('top1_power_frac', ascending=False).to_string())

# Merge into stability
stability = stability.join(fft_raw_df[['top1_period_ticks', 'top1_power_frac']], how='left')
stability = stability.join(fft_df[['dominant_power_frac']], how='left')


Raw FFT (day 2, detrended) — products with top-1 frequency capturing >30% of power:
                               top1_freq  top1_period_ticks  top1_power_frac
product                                                                     
ROBOT_IRONING                     0.0001       10000.000000         0.852463
GALAXY_SOUNDS_PLANETARY_RINGS     0.0001       10000.000000         0.826837
SLEEP_POD_SUEDE                   0.0001       10000.000000         0.778282
TRANSLATOR_ECLIPSE_CHARCOAL       0.0001       10000.000000         0.773588
OXYGEN_SHAKE_GARLIC               0.0001       10000.000000         0.736018
OXYGEN_SHAKE_CHOCOLATE            0.0001       10000.000000         0.716826
PANEL_1X4                         0.0001       10000.000000         0.700372
TRANSLATOR_VOID_BLUE              0.0001       10000.000000         0.670156
GALAXY_SOUNDS_SOLAR_FLAMES        0.0001       10000.000000         0.652486
ROBOT_LAUNDRY                     0.0001       10000.000000         

## 3. Volatility clustering: squared-returns autocorrelation

ARCH-like effects: if |r_t|^2 is autocorrelated, volatility comes in bursts. This matters for risk sizing (though we do not specify rules here). Compute ACF of squared returns at lags 1, 2, 5, 10 for each product.

In [11]:
def acf_at_lag(series, lag):
    """Pearson correlation between series and series shifted by lag."""
    x = series[:-lag]
    y = series[lag:]
    if len(x) < 20:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

vol_records = []
for prod in PRODUCTS:
    # Use all 3 days concatenated
    series_list = []
    for d in days:
        sub = dfs[d][dfs[d]['product'] == prod].sort_values('timestamp')['mid_price'].values
        if len(sub) > 1:
            rets = np.diff(np.log(sub + 1e-9))
            series_list.append(rets)
    if not series_list:
        continue
    rets = np.concatenate(series_list)
    sq_rets = rets**2
    abs_rets = np.abs(rets)
    
    row = {'product': prod}
    for lag in [1, 2, 5, 10]:
        row[f'sq_acf_lag{lag}'] = acf_at_lag(sq_rets, lag)
        row[f'abs_acf_lag{lag}'] = acf_at_lag(abs_rets, lag)
    # Simple vol summary
    row['ret_std'] = rets.std()
    row['ret_kurtosis'] = pd.Series(rets).kurtosis()
    vol_records.append(row)

vol_df = pd.DataFrame(vol_records).set_index('product')
print('Squared-returns ACF (lag 1 and 5) — top products by clustering strength:')
print(vol_df[['sq_acf_lag1', 'sq_acf_lag5', 'abs_acf_lag1', 'ret_std', 'ret_kurtosis']]
      .sort_values('sq_acf_lag1', ascending=False).to_string())

Squared-returns ACF (lag 1 and 5) — top products by clustering strength:
                               sq_acf_lag1  sq_acf_lag5  abs_acf_lag1   ret_std  ret_kurtosis
product                                                                                      
ROBOT_DISHES                      0.262083     0.116523      0.202176  0.001703     20.074071
OXYGEN_SHAKE_CHOCOLATE            0.239487     0.079249      0.074812  0.001117     10.774505
OXYGEN_SHAKE_EVENING_BREATH       0.239091     0.047451      0.066206  0.001170     10.495136
ROBOT_IRONING                     0.215613     0.134561      0.054039  0.001177      8.834272
PEBBLES_XS                        0.056533     0.043158      0.057572  0.002141      0.321057
PEBBLES_XL                        0.038738     0.030760      0.031765  0.002359      0.238524
PEBBLES_S                         0.028003     0.014092      0.021291  0.001705      0.104339
SNACKPACK_STRAWBERRY              0.016669     0.008085      0.011509  0.000761  

In [12]:
# Merge vol_df into stability
stability = stability.join(vol_df[['sq_acf_lag1', 'sq_acf_lag5', 'ret_std', 'ret_kurtosis']], how='left')

# Plot: volatility clustering heatmap across categories
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: sq_acf_lag1 bar chart per product
ax = axes[0]
sq_sorted = vol_df['sq_acf_lag1'].sort_values(ascending=False)
colors_bar = [cat_colors[get_cat(p)] for p in sq_sorted.index]
ax.barh(range(len(sq_sorted)), sq_sorted.values, color=colors_bar, alpha=0.8)
ax.set_yticks(range(len(sq_sorted)))
ax.set_yticklabels([p.replace('GALAXY_SOUNDS_','GS_').replace('SLEEP_POD_','SP_')
                    .replace('OXYGEN_SHAKE_','OX_').replace('TRANSLATOR_','TR_')
                    .replace('SNACKPACK_','SN_') for p in sq_sorted.index], fontsize=6)
ax.axvline(0.1, color='red', linestyle='--', alpha=0.6, label='ACF=0.1')
ax.set_xlabel('ACF of squared returns at lag 1')
ax.set_title('Volatility clustering (sq. return ACF lag 1)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

# Right: ret_kurtosis (fat tails → more volatile clustering)
ax2 = axes[1]
kurt_sorted = vol_df['ret_kurtosis'].sort_values(ascending=False)
colors_bar2 = [cat_colors[get_cat(p)] for p in kurt_sorted.index]
ax2.barh(range(len(kurt_sorted)), kurt_sorted.values, color=colors_bar2, alpha=0.8)
ax2.set_yticks(range(len(kurt_sorted)))
ax2.set_yticklabels([p.replace('GALAXY_SOUNDS_','GS_').replace('SLEEP_POD_','SP_')
                     .replace('OXYGEN_SHAKE_','OX_').replace('TRANSLATOR_','TR_')
                     .replace('SNACKPACK_','SN_') for p in kurt_sorted.index], fontsize=6)
ax2.axvline(3.0, color='red', linestyle='--', alpha=0.6, label='Kurtosis=3 (normal)')
ax2.set_xlabel('Excess kurtosis of returns')
ax2.set_title('Fat tails of returns (excess kurtosis)')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/08_vol_clustering.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved vol clustering')

Saved vol clustering


## 4. Category-level regime summary

Aggregate all regime signals to the category level to support the triage rubric.

In [13]:
# Category-level aggregation
cat_summary = stability.groupby('category').agg(
    mean_cv_avg=('mean_cv', 'mean'),
    ar1_mean_avg=('ar1_mean', 'mean'),
    ar1_abs_avg=('ar1_mean', lambda x: x.abs().mean()),
    ar1_range_avg=('ar1_range', 'mean'),
    std_mean_avg=('std_mean', 'mean'),
    intraday_amp_avg=('intraday_amplitude', 'mean'),
    sq_acf_lag1_avg=('sq_acf_lag1', 'mean'),
    n_stable=('stability', lambda x: (x == 'STABLE').sum()),
    n_drifting=('stability', lambda x: (x == 'DRIFTING').sum()),
).round(4)

print('Category-level regime summary:')
print(cat_summary.to_string())

Category-level regime summary:
               mean_cv_avg  ar1_mean_avg  ar1_abs_avg  ar1_range_avg  std_mean_avg  intraday_amp_avg  sq_acf_lag1_avg  n_stable  n_drifting
category                                                                                                                                   
GALAXY_SOUNDS       0.0476       -0.0101       0.0101         0.0168      393.0844          764.3465           0.0035         0           3
MICROCHIP           0.1172       -0.0090       0.0090         0.0127      486.2261         1009.4125           0.0051         0           5
OXYGEN_SHAKE        0.0481       -0.0399       0.0399         0.0502      399.5007          879.8435           0.0951         0           4
PANEL               0.0603       -0.0043       0.0043         0.0208      386.8841          717.7944          -0.0028         0           5
PEBBLES             0.1039        0.0006       0.0092         0.0135      640.5800         1427.1478           0.0260         0  

In [14]:
# Final triage table for this notebook
# Assign regime triage call per product

def regime_triage(row):
    """
    'likely exploitable': stable stats (CV<0.005, ar1_range<0.1) + |ar1_mean|>0.3
    'probably tradable': stable + moderate AR(1) 0.1-0.3, or mostly stable
    'probably noise': drifting, ar1 near 0, no intraday structure
    """
    ar1_abs = abs(row['ar1_mean'])
    stable = row['stability'] in ('STABLE', 'MOSTLY_STABLE')
    if stable and ar1_abs > 0.3:
        return 'likely exploitable'
    elif stable and ar1_abs > 0.1:
        return 'probably tradable'
    elif row['intraday_amplitude'] > 5.0 and stable:
        return 'probably tradable'
    elif row['stability'] == 'STABLE' and ar1_abs <= 0.1:
        return 'probably tradable'  # Stable = good for FV strategies even without AR signal
    else:
        return 'probably noise'

stability['regime_triage'] = stability.apply(regime_triage, axis=1)

print('Regime triage counts:')
print(stability['regime_triage'].value_counts())

print('\n--- LIKELY EXPLOITABLE ---')
le = stability[stability['regime_triage']=='likely exploitable'][['category','price_mean','std_mean','ar1_mean','mean_cv','ar1_range','stability','intraday_amplitude']]
print(le.to_string())

print('\n--- PROBABLY TRADABLE ---')
pt = stability[stability['regime_triage']=='probably tradable'][['category','price_mean','std_mean','ar1_mean','mean_cv','ar1_range','stability','intraday_amplitude']]
print(pt.to_string())

print('\n--- PROBABLY NOISE ---')
pn = stability[stability['regime_triage']=='probably noise'][['category','price_mean','std_mean','ar1_mean','mean_cv','ar1_range','stability']]
print(pn.to_string())

Regime triage counts:
regime_triage
probably noise       41
probably tradable     9
Name: count, dtype: int64

--- LIKELY EXPLOITABLE ---
Empty DataFrame
Columns: [category, price_mean, std_mean, ar1_mean, mean_cv, ar1_range, stability, intraday_amplitude]
Index: []

--- PROBABLY TRADABLE ---
                                  category    price_mean    std_mean  ar1_mean   mean_cv  ar1_range      stability  intraday_amplitude
GALAXY_SOUNDS_DARK_MATTER    GALAXY_SOUNDS  10226.661817  298.245643 -0.011471  0.016622   0.025965  MOSTLY_STABLE          496.304667
GALAXY_SOUNDS_SOLAR_FLAMES   GALAXY_SOUNDS  11092.571700  422.604221 -0.012007  0.015234   0.021098  MOSTLY_STABLE          711.742000
OXYGEN_SHAKE_EVENING_BREATH   OXYGEN_SHAKE   9271.895000  353.065193 -0.111781  0.014510   0.085569  MOSTLY_STABLE          800.648667
SLEEP_POD_LAMB_WOOL              SLEEP_POD  10701.441717  371.408523  0.003743  0.018692   0.024394  MOSTLY_STABLE          640.949333
SNACKPACK_CHOCOLATE            

In [15]:
# Full summary table with all key metrics
summary_cols = ['category', 'price_mean', 'std_mean', 'ar1_mean', 'mean_cv', 'ar1_range',
                'stability', 'intraday_amplitude', 'sq_acf_lag1', 'ret_kurtosis',
                'top1_period_ticks', 'top1_power_frac', 'regime_triage']
final_table = stability[summary_cols].sort_values(['regime_triage', 'ar1_mean'])
print('FULL REGIME SUMMARY TABLE:')
print(final_table.to_string())

FULL REGIME SUMMARY TABLE:
                                    category    price_mean     std_mean  ar1_mean   mean_cv  ar1_range      stability  intraday_amplitude  sq_acf_lag1  ret_kurtosis  top1_period_ticks  top1_power_frac      regime_triage
ROBOT_IRONING                          ROBOT   8701.570267   427.667547 -0.116738  0.088063   0.075410       DRIFTING          917.280000     0.215613      8.834272       10000.000000         0.852463     probably noise
ROBOT_DISHES                           ROBOT  10018.314900   280.190595 -0.097614  0.058518   0.289736       DRIFTING          658.173333     0.262083     20.074071       10000.000000         0.464003     probably noise
OXYGEN_SHAKE_CHOCOLATE          OXYGEN_SHAKE   9556.879133   380.193627 -0.075971  0.046979   0.111020       DRIFTING          752.929000     0.239487     10.774505       10000.000000         0.716826     probably noise
MICROCHIP_SQUARE                   MICROCHIP  13594.748300   772.154518 -0.022137  0.148741  

In [16]:
# Per-day mean trajectory plot for products with highest mean_cv (most drifting)
top_drift = stability.sort_values('mean_cv', ascending=False).head(10).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, prod in enumerate(top_drift):
    ax = axes[i]
    for d, col in zip(days, ['blue', 'orange', 'green']):
        sub = dfs[d][dfs[d]['product'] == prod].sort_values('timestamp')
        ax.plot(sub['timestamp'].values, sub['mid_price'].values, color=col, alpha=0.6, linewidth=0.5, label=f'Day {d}')
    ax.set_title(prod.replace('GALAXY_SOUNDS_','GS_').replace('SLEEP_POD_','SP_')
                 .replace('OXYGEN_SHAKE_','OX_').replace('TRANSLATOR_','TR_'), fontsize=8)
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('Mid price')
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

plt.suptitle('Most drifting products — mid_price time-series across 3 days', fontsize=12)
plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/08_drifting_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved drifting time-series')

Saved drifting time-series


In [17]:
# Summary statistics to verify AGENT_BRIEF triage boundaries
print('=== REGIME ANALYSIS SUMMARY ===')
print(f'Total products: {len(PRODUCTS)}')
print(f"\nStability breakdown:")
print(stability['stability'].value_counts())
print(f"\nRegime triage:")
print(stability['regime_triage'].value_counts())
print(f"\nProducts with |AR(1)| > 0.3 (any day):")
for p in PRODUCTS:
    row = stability.loc[p]
    if abs(row['ar1_mean']) > 0.3:
        print(f"  {p}: ar1_mean={row['ar1_mean']:.4f}, ar1_range={row['ar1_range']:.4f}, stability={row['stability']}")
print(f"\nProducts with mean_cv > 0.02 (strongly drifting):")
for p in PRODUCTS:
    row = stability.loc[p]
    if row['mean_cv'] > 0.02:
        print(f"  {p}: mean_cv={row['mean_cv']:.4f}, category={row['category']}")
print(f"\nProducts with high intraday amplitude (>10):")
for p in PRODUCTS:
    row = stability.loc[p]
    if row.get('intraday_amplitude', 0) > 10:
        print(f"  {p}: intraday_amplitude={row['intraday_amplitude']:.2f}, category={row['category']}")

=== REGIME ANALYSIS SUMMARY ===
Total products: 50

Stability breakdown:
stability
DRIFTING         41
MOSTLY_STABLE     8
STABLE            1
Name: count, dtype: int64

Regime triage:
regime_triage
probably noise       41
probably tradable     9
Name: count, dtype: int64

Products with |AR(1)| > 0.3 (any day):

Products with mean_cv > 0.02 (strongly drifting):
  GALAXY_SOUNDS_BLACK_HOLES: mean_cv=0.0885, category=GALAXY_SOUNDS
  GALAXY_SOUNDS_PLANETARY_RINGS: mean_cv=0.0748, category=GALAXY_SOUNDS
  GALAXY_SOUNDS_SOLAR_WINDS: mean_cv=0.0430, category=GALAXY_SOUNDS
  MICROCHIP_CIRCLE: mean_cv=0.0358, category=MICROCHIP
  MICROCHIP_OVAL: mean_cv=0.2197, category=MICROCHIP
  MICROCHIP_RECTANGLE: mean_cv=0.0857, category=MICROCHIP
  MICROCHIP_SQUARE: mean_cv=0.1487, category=MICROCHIP
  MICROCHIP_TRIANGLE: mean_cv=0.0959, category=MICROCHIP
  OXYGEN_SHAKE_CHOCOLATE: mean_cv=0.0470, category=OXYGEN_SHAKE
  OXYGEN_SHAKE_GARLIC: mean_cv=0.0781, category=OXYGEN_SHAKE
  OXYGEN_SHAKE_MINT: mean